# MMGCN · MELD — Directed Cross-Modal Edge Gating, Modality Corruption Robustness, and Minority-Class Calibration (YZM5110)

**Objective:** Extending [MMGCN](https://arxiv.org/abs/2107.06779) (Hu et al., ACL-IJCNLP 2021) on the **MELD** dataset to investigate:
1. **Primary Improvement (Cross-Modal Edge Gate & Corruption Propagation Robustness):** Introducing a learnable **Directed Cross-Modal Sigmoid Gate (`GatedMM_GCN`)** exclusively on inter-modal edges ($m \neq n$) alongside readout reliability modulation, and measuring robustness when acoustic ($a$) or visual ($v$) features are corrupted.
2. **Recurring Failure Mitigation (`fear` & `disgust` F1 = 0% Collapse):**
   - **Intervention 2A (Post-Hoc Logit Adjustment / Prior Calibration):** Calibrating class priors at inference time via $\tau$ selected **exclusively on the validation split** without altering model weights (reported side-by-side with standard inference for every variant).
   - **Intervention 2B (In-Training Class-Weighted Loss Ablation — `G1_CW`):** Augmenting MELD's unweighted `FocalLoss` with dampened square-root inverse class frequency weights.

### Experimental Matrix (Matched Conditions)
| Variant | Directed Cross-Modal Gate | Training Corruption Augmentation | Loss Function | Scientific Role |
| :--- | :---: | :---: | :---: | :--- |
| **B0** | No (Original) | No | Original FocalLoss | Reproduced Baseline |
| **B1** | No (Original) | Yes | Original FocalLoss | **Control** (isolates effect of training augmentation alone) |
| **G0** | **Yes (Directed Sigmoid)** | No | Original FocalLoss | **Ablation** (isolates effect of cross-modal gate on clean training) |
| **G1** | **Yes (Directed Sigmoid)** | Yes | Original FocalLoss | **Proposed Model** (Directed Gate + Corruption Augmentation) |
| **G1_CW** | **Yes (Directed Sigmoid)** | Yes | Class-Weighted FocalLoss | **Ablation (2B)** (In-training minority class weighting) |

> **Note:** All outputs, CSV tables, JSON summaries, and figures are saved to `/content/mmgcn_meld_experiments`.

## 1. Verify Colab GPU and Runtime Environment

In [ ]:
import sys, platform, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA (torch):', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Please enable a CUDA GPU: Runtime -> Change runtime type -> T4 GPU'
DEVICE = torch.device('cuda')

## 2. Install Dependencies and Clone Official Repository at Pinned Commit
Pins commit `85732d984f70c1c84dd47c81aa97f1271397b899` and applies idempotent PyTorch 2.x compatibility patches.

In [ ]:
%pip -q install "torch-geometric==2.8.1" "ipdb>=0.13,<1" "scikit-learn>=1.5" "pandas>=2" "matplotlib>=3"

In [ ]:
from pathlib import Path
import os, subprocess

ROOT = Path('/content/MMGCN')
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/hujingwen6666/MMGCN.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '85732d984f70c1c84dd47c81aa97f1271397b899'], check=True)
os.chdir(ROOT)

# Idempotent compatibility patch for PyTorch 2.x
model_file = ROOT / 'model.py'
src = model_file.read_text(encoding='utf-8')
src = src.replace(
    'label_onehot = torch.zeros([seq_length, labels_length]).cuda().scatter_(1, new_label, 1)',
    'label_onehot = torch.zeros([seq_length, labels_length], device=logits.device).scatter_(1, new_label, 1)',
    1
)
src = src.replace('log_p = F.log_softmax(logits)', 'log_p = F.log_softmax(logits, dim=1)', 1)
lines = src.splitlines()
for i, line in enumerate(lines):
    if 'self.graph_model = MM_GCN(' in line and 'nlayers=64' in line:
        lines[i] = line.replace('nlayers=64', 'nlayers=Deep_GCN_nlayers', 1)
model_file.write_text('\n'.join(lines) + '\n', encoding='utf-8')

mm_file = ROOT / 'model_mm.py'
mm_src = mm_file.read_text(encoding='utf-8')
mm_src = mm_src.replace('adj[idx] = dia_sim', 'adj[idx[0], idx[1]] = dia_sim', 1)
mm_file.write_text(mm_src, encoding='utf-8')

pkl = ROOT / 'MELD_features' / 'MELD_features_raw1.pkl'
assert pkl.exists(), f'MELD feature file not found: {pkl}'
print('Ready. Source commit:', subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip())

## 3. Experiment Configuration
- `VARIANTS_TO_RUN`: List of variants to execute (`['B0', 'B1', 'G0', 'G1', 'G1_CW']` runs the full suite sequentially).
- For a quick 45-second pipeline check, set `SMOKE_TEST = True` (executes 2 epochs × 2 batches per variant). Set `SMOKE_TEST = False` for the full 60-epoch experiments.

In [ ]:
import random, numpy as np

SEED = 27350                      # Official train.py seed (matched across all variants)
GRAPH_LAYERS = 64                 # Effective MMGCN layer count
EPOCHS = 60                       # Full training epochs
BATCH_SIZE = 16
LR = 0.001
WEIGHT_DECAY = 0.0
DROPOUT = 0.4
VALID_RATIO = 0.10

SMOKE_TEST = False                # False for full 60-epoch run; True for quick 2-batch verification
REUSE_B0_IF_EXISTS = True         # Reuse existing B0 checkpoint if present
VARIANTS_TO_RUN = ['B0', 'B1', 'G0', 'G1', 'G1_CW']

# Training-time modality corruption augmentation (for B1, G1, G1_CW)
AUG_PROB = 0.30                   # Base probability parameter for per-utterance audio/visual corruption
AUG_NOISE_STD = 1.0               # Gaussian noise scale in multiples of training feature std

CLASS_NAMES = ['neutral', 'surprise', 'fear', 'sadness', 'joy', 'disgust', 'anger']

OUT = Path('/content/mmgcn_meld_experiments')
OUT.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
print('Variants to run:', VARIANTS_TO_RUN, '| SMOKE_TEST:', SMOKE_TEST)

## 4. MELD Dataset, Dialogue-Level Split, and Training Feature Statistics
- Constructs the exact same dialogue-level 90%/10% train/validation split (`SEED=27350`) as the baseline.
- Computes acoustic and visual feature standard deviations (`AUDIO_STD`, `VISUAL_STD`) and empirical class priors (`CLASS_PRIORS`) **strictly from the training split** (zero data leakage).

In [ ]:
from dataloader import MELDDataset
from torch.utils.data import DataLoader, Subset
from collections import Counter

train_dataset = MELDDataset(str(pkl), train=True)
test_dataset = MELDDataset(str(pkl), train=False)

order = list(range(len(train_dataset)))
random.Random(SEED).shuffle(order)
n_valid = max(1, round(len(order) * VALID_RATIO))
train_indices = order[n_valid:]
valid_indices = order[:n_valid]
assert set(train_indices).isdisjoint(valid_indices)

train_loader = DataLoader(Subset(train_dataset, train_indices), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=train_dataset.collate_fn, num_workers=0)
valid_loader = DataLoader(Subset(train_dataset, valid_indices), batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=train_dataset.collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=test_dataset.collate_fn, num_workers=0)

# Extract class frequencies and audio/visual feature standard deviations from training split only
train_labels_all = []
all_vis, all_aud = [], []
for idx in train_indices:
    textf, visuf, acouf, spk, umask, lbl, vid = train_dataset[idx]
    train_labels_all.extend(lbl.tolist())
    all_vis.append(visuf)
    all_aud.append(acouf)

all_vis_cat = torch.cat(all_vis, dim=0)
all_aud_cat = torch.cat(all_aud, dim=0)
VISUAL_STD = all_vis_cat.std(dim=0, unbiased=False).clamp(min=1e-6).to(DEVICE)
AUDIO_STD = all_aud_cat.std(dim=0, unbiased=False).clamp(min=1e-6).to(DEVICE)

counts = Counter(train_labels_all)
total_train_utt = sum(counts.values())
CLASS_COUNTS = [counts[c] for c in range(7)]
CLASS_PRIORS = torch.tensor([counts[c] / total_train_utt for c in range(7)], dtype=torch.float32, device=DEVICE)
LOG_PRIORS = torch.log(CLASS_PRIORS.clamp(min=1e-8))

# Dampened square-root inverse-frequency class weights
inv_freq = torch.tensor([total_train_utt / (7.0 * counts[c]) for c in range(7)], dtype=torch.float32, device=DEVICE)
CLASS_WEIGHTS_SQRT = torch.sqrt(inv_freq)
CLASS_WEIGHTS_SQRT = CLASS_WEIGHTS_SQRT / CLASS_WEIGHTS_SQRT.mean()

print('Dialogue counts:', {'train': len(train_indices), 'valid': len(valid_indices), 'test': len(test_dataset)})
print('Training utterance distribution:')
for c in range(7):
    print(f'  [{c}] {CLASS_NAMES[c]:8s}: {CLASS_COUNTS[c]:4d} ({100*CLASS_PRIORS[c].item():5.2f}%) | SqrtWeight: {CLASS_WEIGHTS_SQRT[c].item():.3f}')

## 5. Proposed Architecture: Directed Cross-Modal Edge Gate (`GatedMM_GCN`)

### Mechanism and Hypothesis (Mitigating Corruption Propagation)
- **Why not over-smoothing?** MMGCN relies on **GCNII** (`alpha=0.1` initial residual and `theta_l` identity mapping), which already prevents deep over-smoothing across 64 layers.
- **True Vulnerability (Corruption Propagation):** In MMGCN, cross-modal edges between $(a_i, v_i, l_i)$ use fixed angular similarity:
  $$A_{m \to n}(i) = 1 - \frac{\arccos(\cos(h_m(i), h_n(i)))}{\pi}$$
  When acoustic or visual features degrade into orthogonal noise ($\cos \approx 0$), $A_{m \to n}(i) \approx 0.50$, continuously injecting noise into the text node at 50% weight across 64 layers.
- **Proposed Directed Sigmoid Gate:**
  - Applies modality-specific `LayerNorm` $\tilde{h}_m(i) = \text{LN}_m(h_m(i))$ and log-energy $r_n(i) = \log(1 + \|h_n(i)\|_2 / \sqrt{d})$.
  - Computes a directed gate for each of the 6 cross-modal pairs ($m \neq n$):
    $$g_{m \leftarrow n}(i) = \sigma\Big(W_{mn}^{(2)}\,\text{ReLU}\big(W_{mn}^{(1)}[\tilde{h}_m(i) \,\|\, \tilde{h}_n(i) \,\|\, (\tilde{h}_m(i) \odot \tilde{h}_n(i)) \,\|\, s_{mn}(i) \,\|\, r_m(i) \,\|\, r_n(i)] + b_{mn}^{(1)}\big) + b_{mn}^{(2)}\Big)$$
  - Warm-started with $b_{mn}^{(2)} = +2.0$ ($\sigma(2.0) \approx 0.88$) to preserve clean-input performance, and reused at the readout stage to scale audio/visual branches before concatenation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np, math
from model import DialogueGCNModel, FocalLoss
from model_mm import MM_GCN

class WeightedFocalLoss(nn.Module):
    """Class-weighted extension of MMGCN's FocalLoss formula (for Ablation 2B: G1_CW)."""
    def __init__(self, gamma=2.5, class_weights=None, size_average=True):
        super().__init__()
        self.gamma = gamma
        self.class_weights = class_weights
        self.size_average = size_average

    def forward(self, logits, labels):
        labels_length = logits.size(1)
        seq_length = logits.size(0)
        new_label = labels.unsqueeze(1)
        label_onehot = torch.zeros([seq_length, labels_length], device=logits.device).scatter_(1, new_label, 1)
        log_p = F.log_softmax(logits, dim=1)
        pt = label_onehot * log_p
        sub_pt = 1 - pt
        fl = -(sub_pt ** self.gamma) * log_p
        if self.class_weights is not None:
            sample_w = self.class_weights[labels].unsqueeze(1)
            fl = fl * sample_w
        return fl.mean() if self.size_average else fl.sum()


class GatedMM_GCN(MM_GCN):
    """
    MM_GCN extended with learnable directed sigmoid gates strictly on cross-modal edges
    and text-anchored readout reliability modulation.
    When use_edge_gate=False, behavior is 100% identical to original MM_GCN.
    """
    def __init__(self, *args, use_edge_gate=False, init_gate_bias=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_edge_gate = use_edge_gate
        n_dim = kwargs.get('n_dim', 200)
        self.n_dim = n_dim
        if self.use_edge_gate:
            self.ln_modals = nn.ModuleList([nn.LayerNorm(n_dim) for _ in range(3)])
            in_dim = 3 * n_dim + 3
            self.inter_edge_gates = nn.ModuleDict()
            for m in range(3):
                for n in range(3):
                    if m != n:
                        mlp = nn.Sequential(
                            nn.Linear(in_dim, 64),
                            nn.ReLU(),
                            nn.Linear(64, 1)
                        )
                        nn.init.zeros_(mlp[2].weight)
                        nn.init.constant_(mlp[2].bias, init_gate_bias)
                        self.inter_edge_gates[f'{m}_{n}'] = mlp
            self.last_gate_stats = {}

    def compute_inter_gate(self, m, n, modal_m, modal_n, dia_cos_sim):
        hm = self.ln_modals[m](modal_m)
        hn = self.ln_modals[n](modal_n)
        norm_m = torch.log1p(modal_m.norm(p=2, dim=1, keepdim=True) / math.sqrt(self.n_dim))
        norm_n = torch.log1p(modal_n.norm(p=2, dim=1, keepdim=True) / math.sqrt(self.n_dim))
        sim_feat = dia_cos_sim.unsqueeze(1)
        gate_in = torch.cat([hm, hn, hm * hn, sim_feat, norm_m, norm_n], dim=1)
        g = torch.sigmoid(self.inter_edge_gates[f'{m}_{n}'](gate_in)).squeeze(1)
        return g

    def create_big_adj(self, a, v, l, dia_len, modals):
        modal_num = len(modals)
        all_length = l.shape[0] if len(l) != 0 else a.shape[0] if len(a) != 0 else v.shape[0]
        device = l.device if len(l) != 0 else a.device
        adj = torch.zeros((modal_num * all_length, modal_num * all_length), device=device)
        features = [a, v, l]

        gate_sums = {f'{m}_{n}': 0.0 for m in range(3) for n in range(3) if m != n}
        total_utt = max(1, int(all_length))

        start = 0
        for i in range(len(dia_len)):
            L = dia_len[i]
            sub_adjs = []
            for j, x in enumerate(features):
                temp = x[start:start + L]
                vec_length = torch.sqrt(torch.sum(temp.mul(temp), dim=1)).clamp(min=1e-8)
                norm_temp = temp.permute(1, 0) / vec_length
                cos_sim_matrix = torch.sum(torch.matmul(norm_temp.unsqueeze(2), norm_temp.unsqueeze(1)), dim=0)
                cos_sim_matrix = cos_sim_matrix.clamp(-1.0, 1.0) * 0.99999
                sim_matrix = 1.0 - torch.acos(cos_sim_matrix) / np.pi
                sub_adjs.append(sim_matrix)

            dia_idx = np.array(np.diag_indices(L))
            for m in range(modal_num):
                for n in range(modal_num):
                    m_start = start + all_length * m
                    n_start = start + all_length * n
                    if m == n:
                        adj[m_start:m_start + L, n_start:n_start + L] = sub_adjs[m]
                    else:
                        modal1 = features[m][start:start + L]
                        modal2 = features[n][start:start + L]
                        len1 = torch.sqrt(torch.sum(modal1.mul(modal1), dim=1)).clamp(min=1e-8)
                        len2 = torch.sqrt(torch.sum(modal2.mul(modal2), dim=1)).clamp(min=1e-8)
                        normed_modal1 = modal1.permute(1, 0) / len1
                        normed_modal2 = modal2.permute(1, 0) / len2
                        dia_cos_sim = torch.sum(normed_modal1.mul(normed_modal2).permute(1, 0), dim=1)
                        dia_cos_sim = dia_cos_sim.clamp(-1.0, 1.0) * 0.99999
                        dia_sim = 1.0 - torch.acos(dia_cos_sim) / np.pi
                        if self.use_edge_gate:
                            g = self.compute_inter_gate(m, n, modal1, modal2, dia_cos_sim)
                            dia_sim = g * dia_sim
                            gate_sums[f'{m}_{n}'] += float(g.detach().sum().item())
                        idx = dia_idx.copy()
                        idx[0] += m_start
                        idx[1] += n_start
                        adj[idx[0], idx[1]] = dia_sim
            start += L

        if self.use_edge_gate:
            self.last_gate_stats = {k: v / total_utt for k, v in gate_sums.items()}

        d = adj.sum(1).clamp(min=1e-8)
        D = torch.diag(torch.pow(d, -0.5))
        adj = D.mm(adj).mm(D)
        return adj

    def forward(self, a, v, l, dia_len, qmask):
        qmask_cat = torch.cat([qmask[:x, i, :] for i, x in enumerate(dia_len)], dim=0)
        spk_idx = torch.argmax(qmask_cat, dim=-1)
        spk_emb_vector = self.speaker_embeddings(spk_idx)
        if self.use_speaker and 'l' in self.modals:
            l = l + spk_emb_vector
        if self.use_modal:
            emb_idx = torch.LongTensor([0, 1, 2]).to(l.device)
            emb_vector = self.modal_embeddings(emb_idx)
            if 'a' in self.modals:
                a = a + emb_vector[0].reshape(1, -1).expand(a.shape[0], a.shape[1])
            if 'v' in self.modals:
                v = v + emb_vector[1].reshape(1, -1).expand(v.shape[0], v.shape[1])
            if 'l' in self.modals:
                l = l + emb_vector[2].reshape(1, -1).expand(l.shape[0], l.shape[1])

        adj = self.create_big_adj(a, v, l, dia_len, self.modals)
        features_in = torch.cat([a, v, l], dim=0)
        features = self.graph_net(features_in, None, qmask_cat, adj)
        all_length = l.shape[0]

        fa = features[:all_length]
        fv = features[all_length:all_length * 2]
        fl = features[all_length * 2:all_length * 3]

        if self.use_edge_gate:
            g_l_from_a = self.compute_inter_gate(2, 0, l, a, torch.zeros(all_length, device=l.device)).unsqueeze(1)
            g_l_from_v = self.compute_inter_gate(2, 1, l, v, torch.zeros(all_length, device=l.device)).unsqueeze(1)
            fa = fa * g_l_from_a
            fv = fv * g_l_from_v

        self.last_text_repr = fl.detach()

        out_feat = torch.cat([fa, fv, fl], dim=-1)
        if self.return_feature:
            return out_feat
        return F.softmax(self.final_fc(out_feat), dim=-1)


def build_model(use_edge_gate=False, seed=SEED):
    set_seed(seed)
    m = DialogueGCNModel(
        base_model='LSTM', D_m=600, D_g=150, D_p=150, D_e=100, D_h=100,
        D_a=100, graph_hidden_size=100, n_speakers=9, max_seq_len=200,
        window_past=10, window_future=10, n_classes=7,
        listener_state=False, context_attention='general', dropout=DROPOUT,
        nodal_attention=True, no_cuda=False, graph_type='MMGCN', graph_construct='direct',
        use_residue=True, D_m_v=342, D_m_a=300, modals='avl',
        att_type='concat_subsequently', av_using_lstm=False,
        Deep_GCN_nlayers=GRAPH_LAYERS, dataset='MELD', use_speaker=True, use_modal=False,
    )
    gm = GatedMM_GCN(
        a_dim=200, v_dim=200, l_dim=200, n_dim=200, nlayers=GRAPH_LAYERS,
        nhidden=100, nclass=7, dropout=DROPOUT, lamda=0.5, alpha=0.1,
        variant=True, return_feature=True, use_residue=True,
        n_speakers=9, modals=['a', 'v', 'l'], use_speaker=True, use_modal=False,
        use_edge_gate=use_edge_gate, init_gate_bias=2.0
    )
    m.graph_model = gm
    return m.to(DEVICE)

m_base = build_model(use_edge_gate=False)
m_gate = build_model(use_edge_gate=True)
p_base = sum(p.numel() for p in m_base.parameters())
p_gate = sum(p.numel() for p in m_gate.parameters())
print(f'Parameter count — Un-gated (B0/B1): {p_base:,} | Gated (G0/G1): {p_gate:,} (+{p_gate - p_base:,} params, +{100*(p_gate-p_base)/p_base:.2f}%)')
del m_base, m_gate

## 6. Modality Corruption Functions and Post-Hoc Logit Adjustment (Intervention 2A)
- **10 Test-Time Evaluation Conditions:** `clean`, `audio_noise_1` ($\sigma=1.0$), `audio_noise_2` ($\sigma=2.0$), `audio_zero`, `visual_noise_1` ($\sigma=1.0$), `visual_noise_2` ($\sigma=2.0$), `visual_zero`, `av_noise_1`, `av_zero`, and `av_swap`.
- **Post-Hoc Logit Adjustment (Menon et al., ICLR 2021):**
  $$\hat{y} = \arg\max_c \big(\log p(y=c|x) - \tau \log \pi_c\big)$$
  where $\pi_c$ is the training class prior and $\tau \in [0, 1.0]$ is tuned **strictly on the validation split**.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import time, json, csv, copy

def apply_corruption(acouf, visuf, mode='clean', generator=None):
    if mode == 'clean' or mode is None:
        return acouf, visuf
    a = acouf.clone()
    v = visuf.clone()
    a_std = AUDIO_STD.view(1, 1, -1)
    v_std = VISUAL_STD.view(1, 1, -1)

    def randn_like(tensor):
        if generator is not None:
            return torch.randn(tensor.shape, device=tensor.device, dtype=tensor.dtype, generator=generator)
        return torch.randn_like(tensor)

    if mode == 'audio_noise_1':
        a = a + 1.0 * a_std * randn_like(a)
    elif mode == 'audio_noise_2':
        a = a + 2.0 * a_std * randn_like(a)
    elif mode == 'audio_zero':
        a.zero_()
    elif mode == 'visual_noise_1':
        v = v + 1.0 * v_std * randn_like(v)
    elif mode == 'visual_noise_2':
        v = v + 2.0 * v_std * randn_like(v)
    elif mode == 'visual_zero':
        v.zero_()
    elif mode == 'av_noise_1':
        a = a + 1.0 * a_std * randn_like(a)
        v = v + 1.0 * v_std * randn_like(v)
    elif mode == 'av_zero':
        a.zero_()
        v.zero_()
    elif mode == 'av_swap':
        a = torch.flip(a, dims=[0])
        v = torch.flip(v, dims=[0])
    else:
        raise ValueError(f'Unknown corruption mode: {mode}')
    return a, v


def apply_training_augmentation(acouf, visuf, p=AUG_PROB, noise_std=AUG_NOISE_STD):
    a = acouf.clone()
    v = visuf.clone()
    T, B, _ = a.shape
    a_std = AUDIO_STD.view(1, 1, -1)
    v_std = VISUAL_STD.view(1, 1, -1)

    u_a = torch.rand((T, B, 1), device=a.device)
    zero_a = (u_a < (p * 0.25))
    noise_a = (u_a >= (p * 0.25)) & (u_a < (p * 0.65))
    a = torch.where(zero_a, torch.zeros_like(a), a)
    a = torch.where(noise_a, a + noise_std * a_std * torch.randn_like(a), a)

    u_v = torch.rand((T, B, 1), device=v.device)
    zero_v = (u_v < (p * 0.25))
    noise_v = (u_v >= (p * 0.25)) & (u_v < (p * 0.65))
    v = torch.where(zero_v, torch.zeros_like(v), v)
    v = torch.where(noise_v, v + noise_std * v_std * torch.randn_like(v), v)
    return a, v


def compute_metrics_from_logits(all_logits, all_y, tau=0.0):
    if tau != 0.0:
        adj_logits = all_logits - tau * LOG_PRIORS.cpu().unsqueeze(0)
    else:
        adj_logits = all_logits
    preds = adj_logits.argmax(dim=1).tolist()
    y_true = all_y.tolist()
    per_class_f1 = f1_score(y_true, preds, average=None, labels=list(range(7)), zero_division=0) * 100.0
    return {
        'accuracy': round(100.0 * accuracy_score(y_true, preds), 2),
        'weighted_f1': round(100.0 * f1_score(y_true, preds, average='weighted', zero_division=0), 2),
        'macro_f1': round(100.0 * f1_score(y_true, preds, average='macro', zero_division=0), 2),
        'fear_f1': round(float(per_class_f1[2]), 2),
        'disgust_f1': round(float(per_class_f1[5]), 2),
        'per_class_f1': [round(float(x), 2) for x in per_class_f1],
        'predictions': preds,
        'labels': y_true,
    }


def tune_logit_adjustment_tau(val_logits, val_y):
    """Selects tau on the validation set to maximize Macro-F1 and minority F1 within 1.2 points of base W-F1."""
    base_m = compute_metrics_from_logits(val_logits, val_y, tau=0.0)
    best_tau = 0.0
    best_score = base_m['macro_f1']
    for tau in np.linspace(0.05, 1.00, 20):
        m = compute_metrics_from_logits(val_logits, val_y, tau=float(tau))
        if m['weighted_f1'] >= base_m['weighted_f1'] - 1.2:
            score = m['macro_f1'] + 0.25 * (m['fear_f1'] + m['disgust_f1'])
            if score > best_score:
                best_score = score
                best_tau = round(float(tau), 3)
    return best_tau

print('Corruption and Logit Adjustment helpers ready.')

## 7. Training and Evaluation Loop
For each variant:
- Tracks Train/Valid metrics at every epoch and selects the checkpoint with highest validation Weighted-F1.
- Evaluates the selected checkpoint across **clean** and **9 corrupted** test conditions, recording directed gate activations and post-GCN text representation drift $\|h_l^{\text{corrupted}} - h_l^{\text{clean}}\|_2$.

In [ ]:
from tqdm.auto import tqdm

def run_epoch(model, loader, loss_fn, optimizer=None, training=False, augment=False,
              corruption_mode='clean', limit_batches=None, collect_repr=False):
    model.train(training)
    total_weighted_loss, total_count = 0.0, 0
    all_y, all_logits, all_text_repr = [], [], []
    gate_accum = {}
    n_batches = 0

    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(SEED + 999)

    with torch.set_grad_enabled(training):
        for idx, batch in enumerate(loader):
            textf, visuf, acouf, qmask, umask, labels = [b.to(DEVICE) for b in batch[:-1]]
            lengths = umask.sum(dim=1).long().tolist()
            y = torch.cat([labels[j, :n] for j, n in enumerate(lengths)])

            if training and augment:
                acouf, visuf = apply_training_augmentation(acouf, visuf)
            elif not training and corruption_mode != 'clean':
                acouf, visuf = apply_corruption(acouf, visuf, mode=corruption_mode, generator=gen)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits, *_ = model(textf, qmask, umask, lengths, acouf, visuf)
            loss = loss_fn(logits, y)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            total_weighted_loss += float(loss.detach()) * y.numel()
            total_count += y.numel()
            all_y.append(y.detach().cpu())
            all_logits.append(logits.detach().cpu())
            if collect_repr and hasattr(model.graph_model, 'last_text_repr'):
                all_text_repr.append(model.graph_model.last_text_repr.cpu())
            if getattr(model.graph_model, 'use_edge_gate', False):
                for k, v in model.graph_model.last_gate_stats.items():
                    gate_accum[k] = gate_accum.get(k, 0.0) + v
            n_batches += 1
            if limit_batches is not None and idx + 1 >= limit_batches:
                break

    all_y_t = torch.cat(all_y, dim=0)
    all_logits_t = torch.cat(all_logits, dim=0)
    m = compute_metrics_from_logits(all_logits_t, all_y_t, tau=0.0)
    m['loss'] = round(total_weighted_loss / max(1, total_count), 5)
    m['logits'] = all_logits_t
    m['y_tensor'] = all_y_t
    if collect_repr and all_text_repr:
        m['text_repr'] = torch.cat(all_text_repr, dim=0)
    if gate_accum:
        m['gate_stats'] = {k: round(v / n_batches, 4) for k, v in gate_accum.items()}
    return m


VARIANT_SPECS = {
    'B0':    {'use_gate': False, 'augment': False, 'class_weight': False, 'desc': 'Baseline (Original MMGCN)'},
    'B1':    {'use_gate': False, 'augment': True,  'class_weight': False, 'desc': 'Control (Un-gated + Corruption Augmentation)'},
    'G0':    {'use_gate': True,  'augment': False, 'class_weight': False, 'desc': 'Ablation (Directed Cross-Modal Gate Only)'},
    'G1':    {'use_gate': True,  'augment': True,  'class_weight': False, 'desc': 'Proposed (Directed Gate + Corruption Augmentation)'},
    'G1_CW': {'use_gate': True,  'augment': True,  'class_weight': True,  'desc': 'Ablation 2B (Gate + Augmentation + Class-Weighted Loss)'},
}

CORRUPTION_MODES = [
    'clean',
    'audio_noise_1', 'audio_noise_2', 'audio_zero',
    'visual_noise_1', 'visual_noise_2', 'visual_zero',
    'av_noise_1', 'av_zero', 'av_swap'
]

## 8. Run Experiments (Training + Robustness Grid + Minority-Class Calibration)
Executes all configured variants sequentially and prints per-epoch and test/robustness metrics.

In [ ]:
all_summaries = {}
robustness_rows = []
gate_rows = []

limit_b = 2 if SMOKE_TEST else None
n_epochs_run = 2 if SMOKE_TEST else EPOCHS

for vname in VARIANTS_TO_RUN:
    spec = VARIANT_SPECS[vname]
    vdir = OUT / vname
    vdir.mkdir(parents=True, exist_ok=True)
    ckpt_path = vdir / f'best_{vname}.pt'

    print('\n' + '=' * 88)
    print(f'>>> VARIANT: {vname} — {spec["desc"]}')
    print('=' * 88, flush=True)

    model = build_model(use_edge_gate=spec['use_gate'], seed=SEED)
    loss_fn = WeightedFocalLoss(class_weights=CLASS_WEIGHTS_SQRT) if spec['class_weight'] else FocalLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    prev_b0 = Path('/content/mmgcn_meld_results/best_mmgcn_meld.pt')
    loaded_existing = False
    if vname == 'B0' and REUSE_B0_IF_EXISTS and (ckpt_path.exists() or prev_b0.exists()) and not SMOKE_TEST:
        src_ckpt = ckpt_path if ckpt_path.exists() else prev_b0
        ckpt = torch.load(src_ckpt, map_location=DEVICE, weights_only=True)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        best_epoch = ckpt.get('epoch', -1)
        best_val_f1 = ckpt.get('valid_weighted_f1', -1.0)
        train_time_total = 0.0
        peak_mem_mib = 0.0
        loaded_existing = True
        print(f'[INFO] Loaded existing B0 checkpoint ({src_ckpt}, epoch={best_epoch}, val_W-F1={best_val_f1}).')

    if not loaded_existing:
        torch.cuda.reset_peak_memory_stats()
        t_start = time.perf_counter()
        best_val_f1 = -1.0
        best_val_loss = float('inf')
        best_epoch = 1
        best_state = None
        history = []

        for ep in range(1, n_epochs_run + 1):
            ep_t0 = time.perf_counter()
            tr = run_epoch(model, train_loader, loss_fn, optimizer=optimizer,
                           training=True, augment=spec['augment'], limit_batches=limit_b)
            va = run_epoch(model, valid_loader, loss_fn, training=False, limit_batches=limit_b)
            ep_sec = round(time.perf_counter() - ep_t0, 2)

            improved = (va['weighted_f1'] > best_val_f1) or (
                va['weighted_f1'] == best_val_f1 and va['loss'] < best_val_loss
            )
            if improved:
                best_val_f1 = va['weighted_f1']
                best_val_loss = va['loss']
                best_epoch = ep
                best_state = copy.deepcopy(model.state_dict())

            history.append({
                'epoch': ep, 'train_loss': tr['loss'], 'train_wf1': tr['weighted_f1'], 'train_mf1': tr['macro_f1'],
                'valid_loss': va['loss'], 'valid_wf1': va['weighted_f1'], 'valid_mf1': va['macro_f1'],
                'valid_fear_f1': va['fear_f1'], 'valid_disgust_f1': va['disgust_f1'], 'seconds': ep_sec
            })
            if ep % 5 == 0 or ep == 1 or improved or ep == n_epochs_run:
                print(
                    f"[{vname}] Ep {ep:02d}/{n_epochs_run} | "
                    f"Tr W-F1={tr['weighted_f1']:5.2f}% M-F1={tr['macro_f1']:5.2f}% | "
                    f"Va W-F1={va['weighted_f1']:5.2f}% M-F1={va['macro_f1']:5.2f}% "
                    f"(fear={va['fear_f1']:4.1f}, disg={va['disgust_f1']:4.1f}) | {ep_sec:.1f}s"
                    + (' *BEST*' if improved else ''),
                    flush=True
                )

        train_time_total = round(time.perf_counter() - t_start, 2)
        peak_mem_mib = round(torch.cuda.max_memory_allocated() / (1024 ** 2), 1)
        model.load_state_dict(best_state)
        torch.save({'model_state_dict': best_state, 'epoch': best_epoch, 'valid_weighted_f1': best_val_f1}, ckpt_path)
        with open(vdir / 'history.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
            w.writeheader(); w.writerows(history)

    # 1) Tune Post-Hoc Logit Adjustment (tau) on Validation Set
    va_best = run_epoch(model, valid_loader, loss_fn, training=False, limit_batches=limit_b)
    best_tau = tune_logit_adjustment_tau(va_best['logits'], va_best['y_tensor'])
    va_la = compute_metrics_from_logits(va_best['logits'], va_best['y_tensor'], tau=best_tau)

    # 2) Clean Test Evaluation (Standard and Logit-Adjusted)
    t_inf0 = time.perf_counter()
    te_clean = run_epoch(model, test_loader, loss_fn, training=False, corruption_mode='clean',
                         limit_batches=limit_b, collect_repr=True)
    inf_ms_per_utt = round(1000.0 * (time.perf_counter() - t_inf0) / max(1, len(te_clean['labels'])), 3)
    te_clean_la = compute_metrics_from_logits(te_clean['logits'], te_clean['y_tensor'], tau=best_tau)
    clean_text_repr = te_clean['text_repr']

    print(f'\n--- [{vname}] CLEAN TEST RESULTS (Selected Epoch: {best_epoch}, Validation W-F1: {best_val_f1:.2f}%) ---')
    print(f"  Standard (tau=0.00)       -> W-F1: {te_clean['weighted_f1']:.2f}% | Macro-F1: {te_clean['macro_f1']:.2f}% | Acc: {te_clean['accuracy']:.2f}% | fear F1: {te_clean['fear_f1']:.2f}% | disgust F1: {te_clean['disgust_f1']:.2f}%")
    print(f"  Logit-Adjusted (tau={best_tau:.2f}) -> W-F1: {te_clean_la['weighted_f1']:.2f}% | Macro-F1: {te_clean_la['macro_f1']:.2f}% | Acc: {te_clean_la['accuracy']:.2f}% | fear F1: {te_clean_la['fear_f1']:.2f}% | disgust F1: {te_clean_la['disgust_f1']:.2f}%")
    print('  Per-Class F1 (Standard):      ', dict(zip(CLASS_NAMES, te_clean['per_class_f1'])))
    print('  Per-Class F1 (Logit-Adjusted):', dict(zip(CLASS_NAMES, te_clean_la['per_class_f1'])))

    # 3) Modality Corruption (Robustness) Grid
    print(f'\n--- [{vname}] MODALITY CORRUPTION (ROBUSTNESS) EVALUATION ---')
    for cmode in CORRUPTION_MODES:
        te_c = te_clean if cmode == 'clean' else run_epoch(
            model, test_loader, loss_fn, training=False, corruption_mode=cmode,
            limit_batches=limit_b, collect_repr=True
        )
        te_c_la = compute_metrics_from_logits(te_c['logits'], te_c['y_tensor'], tau=best_tau)
        repr_drift = 0.0
        if cmode != 'clean' and 'text_repr' in te_c:
            repr_drift = round(float((te_c['text_repr'] - clean_text_repr).norm(p=2, dim=1).mean().item()), 4)

        row = {
            'variant': vname,
            'corruption': cmode,
            'w_f1': te_c['weighted_f1'],
            'delta_w_f1': round(te_c['weighted_f1'] - te_clean['weighted_f1'], 2),
            'macro_f1': te_c['macro_f1'],
            'acc': te_c['accuracy'],
            'fear_f1': te_c['fear_f1'],
            'disgust_f1': te_c['disgust_f1'],
            'la_w_f1': te_c_la['weighted_f1'],
            'la_macro_f1': te_c_la['macro_f1'],
            'la_fear_f1': te_c_la['fear_f1'],
            'la_disgust_f1': te_c_la['disgust_f1'],
            'text_repr_drift': repr_drift,
        }
        robustness_rows.append(row)
        if 'gate_stats' in te_c:
            g_row = {'variant': vname, 'corruption': cmode, **te_c['gate_stats']}
            gate_rows.append(g_row)
        print(
            f"  {cmode:15s} | W-F1: {row['w_f1']:5.2f}% (Δ={row['delta_w_f1']:+5.2f}) | "
            f"Macro-F1: {row['macro_f1']:5.2f}% | LA Macro-F1: {row['la_macro_f1']:5.2f}% | "
            f"Text Drift: {repr_drift:.4f}"
            + (f" | Gate(l<-a,l<-v): ({te_c['gate_stats']['2_0']:.3f}, {te_c['gate_stats']['2_1']:.3f})" if 'gate_stats' in te_c else '')
        )

    all_summaries[vname] = {
        'variant': vname,
        'description': spec['desc'],
        'best_epoch': best_epoch,
        'valid_weighted_f1': best_val_f1,
        'tuned_tau': best_tau,
        'clean_test_standard': {k: v for k, v in te_clean.items() if k not in ('logits', 'y_tensor', 'text_repr', 'predictions', 'labels')},
        'clean_test_logit_adjusted': {k: v for k, v in te_clean_la.items() if k not in ('predictions', 'labels')},
        'confusion_matrix_standard': confusion_matrix(te_clean['labels'], te_clean['predictions'], labels=list(range(7))).tolist(),
        'confusion_matrix_logit_adjusted': confusion_matrix(te_clean_la['labels'], te_clean_la['predictions'], labels=list(range(7))).tolist(),
        'params': sum(p.numel() for p in model.parameters()),
        'train_time_sec': train_time_total,
        'peak_gpu_mib': peak_mem_mib,
        'inference_ms_per_utt': inf_ms_per_utt,
    }

## 9. Summary Tables and Report Figures
Prints comparative tables, exports CSV/JSON artifacts, and generates high-resolution PNG figures in English.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_rob = pd.DataFrame(robustness_rows)
df_rob.to_csv(OUT / 'robustness_comparison.csv', index=False)
(OUT / 'all_summaries.json').write_text(json.dumps(all_summaries, indent=2, ensure_ascii=False), encoding='utf-8')

if gate_rows:
    df_gate = pd.DataFrame(gate_rows)
    df_gate.to_csv(OUT / 'gate_activations.csv', index=False)

print('=== TABLE 1: CLEAN TEST PERFORMANCE (STANDARD vs. LOGIT-ADJUSTED / MINORITY-CLASS CALIBRATION) ===')
clean_table = []
for vname, s in all_summaries.items():
    st = s['clean_test_standard']
    la = s['clean_test_logit_adjusted']
    clean_table.append({
        'Model': vname,
        'W-F1 (Std)': st['weighted_f1'],
        'Macro-F1 (Std)': st['macro_f1'],
        'fear F1 (Std)': st['fear_f1'],
        'disgust F1 (Std)': st['disgust_f1'],
        'tau (Val)': s['tuned_tau'],
        'W-F1 (LogitAdj)': la['weighted_f1'],
        'Macro-F1 (LogitAdj)': la['macro_f1'],
        'fear F1 (LogitAdj)': la['fear_f1'],
        'disgust F1 (LogitAdj)': la['disgust_f1'],
        'Params': s['params'],
        'Train (s)': s['train_time_sec'],
        'Peak GPU (MiB)': s['peak_gpu_mib'],
    })
df_clean = pd.DataFrame(clean_table)
df_clean.to_csv(OUT / 'clean_and_minority_table.csv', index=False)
print(df_clean.to_string(index=False))

print('\n=== TABLE 2: TEST WEIGHTED-F1 COMPARISON UNDER MODALITY CORRUPTION ===')
piv_wf1 = df_rob.pivot(index='corruption', columns='variant', values='w_f1').reindex(CORRUPTION_MODES)
print(piv_wf1.to_string())

print('\n=== TABLE 3: POST-GCN TEXT REPRESENTATION DRIFT ||h_l(corrupted) - h_l(clean)||_2 (LOWER IS BETTER) ===')
piv_drift = df_rob.pivot(index='corruption', columns='variant', values='text_repr_drift').reindex(CORRUPTION_MODES)
print(piv_drift.to_string())

if gate_rows:
    print('\n=== TABLE 4: MEAN DIRECTED CROSS-MODAL GATE ACTIVATIONS (2_0: l<-a, 2_1: l<-v) ===')
    print(df_gate.to_string(index=False))

# Figure 1: Robustness comparison across modality corruption scenarios
fig, ax = plt.subplots(figsize=(11, 4.8))
piv_wf1.plot(kind='bar', ax=ax)
ax.set_ylabel('Test Weighted-F1 (%)')
ax.set_xlabel('Modality Corruption Scenario')
ax.set_title('MELD: Robustness Comparison Under Audio/Visual Modality Corruption')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=30, ha='right')
fig.tight_layout()
fig.savefig(OUT / 'fig1_robustness_wf1.png', dpi=160)
plt.show()
plt.close(fig)

# Figure 2: Minority class (fear & disgust) and Macro-F1 calibration
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(df_clean))
w = 0.2
ax.bar(x - 1.5*w, df_clean['Macro-F1 (Std)'], width=w, label='Macro-F1 (Standard)')
ax.bar(x - 0.5*w, df_clean['Macro-F1 (LogitAdj)'], width=w, label='Macro-F1 (Logit-Adjusted)')
ax.bar(x + 0.5*w, df_clean['fear F1 (LogitAdj)'], width=w, label='fear F1 (Logit-Adjusted)')
ax.bar(x + 1.5*w, df_clean['disgust F1 (LogitAdj)'], width=w, label='disgust F1 (Logit-Adjusted)')
ax.set_xticks(x)
ax.set_xticklabels(df_clean['Model'])
ax.set_ylabel('F1 Score (%)')
ax.set_title('Minority-Class (fear & disgust) and Macro-F1 Recovery on MELD')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'fig2_minority_f1.png', dpi=160)
plt.show()
plt.close(fig)

## 10. Package All Report Artifacts as a ZIP Archive
Downloads all CSV tables, JSON summaries, and PNG figures (excluding large `.pt` model checkpoints).

In [ ]:
import shutil, zipfile

zip_path = Path('/content/MMGCN_MELD_All_Experiments_Report_Data.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix != '.pt':
            zf.write(p, p.relative_to(OUT))
print('Report data archive ready:', zip_path, f'({round(zip_path.stat().st_size/1024, 1)} KB)')
from google.colab import files
files.download(str(zip_path))